<a href="https://colab.research.google.com/github/mumtajshagul/-Application-to-Manage-the-Mall-CRM/blob/main/output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install torch torchvision torchaudio matplotlib scikit-learn opencv-python

In [ ]:
!install torch torchvision torchaudio matplotlib scikit-learn opencv-python
import torch
import torchvision
from torch import nn
from torchvision import transforms

# Define the model
def get_model(num_classes=2):
    model = torchvision.models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

# Define image transformatpipions
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

# Load the dataset
train_dataset = ImageFolder(root='/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Train', transform=transform)
test_dataset = ImageFolder(root='/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

from sklearn.metrics import accuracy_score

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


import cv2
import numpy as np

def predict_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(image)
        _, prediction = torch.max(output, 1)

    return 'Defective' if prediction.item() == 0 else 'Non-Defective'

# Example usage
print(predict_image('path_to_image.jpg'))

KeyboardInterrupt: 

In [ ]:
import os
print(os.path.exists('path_to_image.jpg'))


False


In [ ]:
import os

def predict_folder(folder_path):
    results = {}  # store results for each image

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # Skip folders
        if os.path.isdir(file_path):
            continue

        # Check image extensions
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            try:
                prediction = predict_image(file_path)
                results[filename] = prediction
            except Exception as e:
                results[filename] = f"Error: {e}"

    return results


# Example usage
folder_path = '/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Test/Defective'
outputs = predict_folder(folder_path)

for img, result in outputs.items():
    print(f"{img}: {result}")


IMG_20201114_100159.jpg: Defective
IMG_20201114_101124.jpg: Defective
IMG_20201114_100209.jpg: Defective
IMG_20201114_103110.jpg: Defective
IMG_20201114_102203.jpg: Defective
IMG_20201114_101200.jpg: Defective
IMG_20201114_102819.jpg: Defective
IMG_20201114_102222.jpg: Defective
IMG_20201211_121713.jpg: Defective
IMG_20201211_121712_1.jpg: Defective
IMG_20201211_121712.jpg: Defective


In [ ]:
import os

def predict_folder(folder_path):
    results = {}  # store results for each image

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # Skip folders
        if os.path.isdir(file_path):
            continue

        # Check image extensions
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            try:
                prediction = predict_image(file_path)
                results[filename] = prediction
            except Exception as e:
                results[filename] = f"Error: {e}"

    return results


# Example usage
folder_path = '/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Test/Non defective'
outputs = predict_folder(folder_path)

for img, result in outputs.items():
    print(f"{img}: {result}")


IMG_20201114_100358.jpg: Non-Defective
IMG_20201114_100344.jpg: Non-Defective
IMG_20201114_100023.jpg: Non-Defective
IMG_20201114_101756.jpg: Defective
IMG_20201114_101907.jpg: Defective
IMG_20201114_101633.jpg: Defective
IMG_20201114_102431.jpg: Defective
IMG_20201114_102710.jpg: Non-Defective
IMG_20201114_102909.jpg: Defective
IMG_20201114_102253.jpg: Defective
IMG_20201114_102945.jpg: Defective


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# ---------------------------------------
# 1. ENTER YOUR ACTUAL & PREDICTED VALUES
# ---------------------------------------
# Example (change these to your values)
actual = np.array([0, 1, 0, 1, 1, 0])
predicted = np.array([0, 1, 1, 1, 0, 0])

# ---------------------------------------
# 2. CONFUSION MATRIX
# ---------------------------------------
cm = confusion_matrix(actual, predicted, labels=[0, 1])

print("Confusion Matrix (00, 01, 10, 11):")
print(cm)

# cm structure:
# cm[0][0] = 00
# cm[0][1] = 01
# cm[1][0] = 10
# cm[1][1] = 11


# ---------------------------------------
# 3. NORMALIZED CONFUSION MATRIX
# ---------------------------------------
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

print("\nNormalized Confusion Matrix:")
print(np.round(cm_normalized, 2))


# ---------------------------------------
# 4. COUNT OF EACH PAIR (00,01,10,11)
# ---------------------------------------
N00 = cm[0][0]
N01 = cm[0][1]
N10 = cm[1][0]
N11 = cm[1][1]

print("\nTotal Occurrences:")
print(f"00 = {N00}")
print(f"01 = {N01}")
print(f"10 = {N10}")
print(f"11 = {N11}")


Confusion Matrix (00, 01, 10, 11):
[[2 1]
 [1 2]]

Normalized Confusion Matrix:
[[0.67 0.33]
 [0.33 0.67]]

Total Occurrences:
00 = 2
01 = 1
10 = 1
11 = 2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torch torchvision



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# ---------------------------
# DEVICE SETUP
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", device)

# ---------------------------
# TRANSFORMS
# ---------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ---------------------------
# LOAD DATASET
# ---------------------------
train_dir = "/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Train"
test_dir  = "/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Test"

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
print("Classes =", class_names)

# ---------------------------
# LOAD RESNET-50
# ---------------------------
model = models.resnet50(pretrained=True)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Replace last layer for 2 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

# ---------------------------
# LOSS + OPTIMIZER
# ---------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ---------------------------
# TRAINING LOOP
# ---------------------------
EPOCHS = 5
print("\nTraining Started...\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_loader):.4f}")

print("\nTraining Finished!")

# ---------------------------
# SAVE MODEL
# ---------------------------
save_path = "/content/drive/MyDrive/railway-track-fault-detection/model.pth"
torch.save(model, save_path)

print("\nModel saved successfully at:")
print(save_path)


DEVICE = cpu
Classes = ['Defective', 'Non defective']


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:03<00:00, 30.4MB/s]



Training Started...

Epoch [1/5] Loss: 0.6115
Epoch [2/5] Loss: 0.5111
Epoch [3/5] Loss: 0.4413
Epoch [4/5] Loss: 0.4167
Epoch [5/5] Loss: 0.4042

Training Finished!

Model saved successfully at:
/content/drive/MyDrive/railway-track-fault-detection/model.pth


In [ ]:
import torch
import torchvision
from torch import nn

def get_model(num_classes=2):
    model = torchvision.models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model



In [ ]:
model = get_model(num_classes=2)
model.load_state_dict(torch.load("model.pt"))
model.eval()
print("Model loaded successfully")



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model loaded successfully


In [ ]:
from PIL import Image
import torch
import torch.nn.functional as F

# Your test image path
img_path = "/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Test/Non defective/IMG_20201114_100358.jpg"

# Load the image
img = Image.open(img_path).convert("RGB")

# Apply the same transform used for training
img_t = transform(img).unsqueeze(0).to(device)

# Predict
with torch.no_grad():
    outputs = model(img_t)
    _, predicted = torch.max(outputs, 1)

# Print results
print("Predicted class index:", predicted.item())
print("Predicted label:", class_names[predicted.item()])


Predicted class index: 1
Predicted label: Non defective


In [ ]:
import os
from PIL import Image
import torch
import pandas as pd

# -----------------------------
# 1) FOLDER PATH
# -----------------------------
test_folder = "/content/drive/MyDrive/railway-track-fault-detection/railway-track-fault-detection/Test/Non defective"

# -----------------------------
# 2) GET ALL TEST IMAGES
# -----------------------------
image_files = sorted([
    f for f in os.listdir(test_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print("Found images:", image_files)

# -----------------------------
# 3) PREDICT FUNCTION
# -----------------------------
def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_t = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_t)
        _, predicted = torch.max(outputs, 1)

    return predicted.item()

# -----------------------------
# 4) COLLECT RESULTS
# -----------------------------
results = []

for imgname in image_files:
    full_path = os.path.join(test_folder, imgname)

    try:
        pred = predict_image(full_path)
        results.append({
            "Image": imgname,
            "Actual": 1,              # Non-Defective folder → actual = 1
            "Predicted": pred
        })
    except Exception as e:
        results.append({
            "Image": imgname,
            "Actual": 1,
            "Predicted": f"Error: {e}"
        })

# -----------------------------
# 5) CREATE DATAFRAME
# -----------------------------
df = pd.DataFrame(results)

# -----------------------------
# 6) SEPARATE VALID NUMERIC PREDICTIONS
# -----------------------------
valid_df = df[df["Predicted"].apply(lambda x: isinstance(x, int))]

TP = len(valid_df[(valid_df.Actual == 1) & (valid_df.Predicted == 1)])
TN = len(valid_df[(valid_df.Actual == 0) & (valid_df.Predicted == 0)])
FP = len(valid_df[(valid_df.Actual == 1) & (valid_df.Predicted == 0)])
FN = len(valid_df[(valid_df.Actual == 0) & (valid_df.Predicted == 1)])

total = len(valid_df)
accuracy = (TP + TN) / total * 100 if total > 0 else 0

# -----------------------------
# 7) PRINT RESULTS
# -----------------------------
print("\nPER IMAGE RESULTS:")
print(df)

print("\nSUMMARY:")
print("Valid predictions :", total)
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print(f"Accuracy: {accuracy:.2f}%")

# -----------------------------
# 8) SHOW ERRORS
# -----------------------------
error_df = df[df["Predicted"].apply(lambda x: not isinstance(x, int))]
if len(error_df) > 0:
    print("\nImages with prediction errors:")
    print(error_df)


Found images: ['IMG_20201114_100023.jpg', 'IMG_20201114_100344.jpg', 'IMG_20201114_100358.jpg', 'IMG_20201114_101633.jpg', 'IMG_20201114_101756.jpg', 'IMG_20201114_101907.jpg', 'IMG_20201114_102253.jpg', 'IMG_20201114_102431.jpg', 'IMG_20201114_102710.jpg', 'IMG_20201114_102909.jpg', 'IMG_20201114_102945.jpg']

PER IMAGE RESULTS:
                      Image  Actual  Predicted
0   IMG_20201114_100023.jpg       1          1
1   IMG_20201114_100344.jpg       1          1
2   IMG_20201114_100358.jpg       1          1
3   IMG_20201114_101633.jpg       1          1
4   IMG_20201114_101756.jpg       1          1
5   IMG_20201114_101907.jpg       1          1
6   IMG_20201114_102253.jpg       1          1
7   IMG_20201114_102431.jpg       1          1
8   IMG_20201114_102710.jpg       1          1
9   IMG_20201114_102909.jpg       1          1
10  IMG_20201114_102945.jpg       1          1

SUMMARY:
Valid predictions : 11
TP: 11
TN: 0
FP: 0
FN: 0
Accuracy: 100.00%
